# Exploratory Data Analysis of Data Science Program (WQU) Applicants Using MongoDB, Pandas, and Plotly

## Pengenalan koneksi MongoDB

1. Menghubungkan Python ke server MongoDB lokal, lalu menampilkan daftar database (admin, air-quality, config, local, wqu-abtest)

In [13]:
# Minimal PyMongo connection example (toy demonstration)
import pandas as pd
from wqulibs.database import reset
from pymongo import MongoClient

MONGODB_HOST = "localhost"

# Connect to the local MongoDB server
demo_client = MongoClient(host=MONGODB_HOST, port=27017)
print("Type of client:", type(demo_client))

# List available databases
print("Databases:", demo_client.list_database_names())

Type of client: <class 'pymongo.synchronous.mongo_client.MongoClient'>
Databases: ['admin', 'air-quality', 'config', 'local', 'wqu-abtest']


2. Mengakses database wqu-abtest dan koleksi ds-applicants

In [14]:
# Access a database and collection
demo_db = demo_client["wqu-abtest"]
demo_collection = demo_db["ds-applicants"]
print("Collection type:", type(demo_collection))

Collection type: <class 'pymongo.synchronous.collection.Collection'>


3. Mereset koleksi ke kondisi awal (fungsi dari wqulibs)

In [15]:
reset(demo_collection)

4. Menghitung total dokumen (5.025) dan melihat struktur satu dokumen (field: nama, email, tanggal lahir, gender, pendidikan, negara, status quiz)

In [16]:
# Count all documents in the collection
n = demo_collection.count_documents({})
print("Total documents:", n)

# Retrieve one document
one_doc = demo_collection.find_one({})
print("Document type:", type(one_doc))
print("Document keys:", list(one_doc.keys()))

Total documents: 5025
Document type: <class 'dict'>
Document keys: ['_id', 'createdAt', 'firstName', 'lastName', 'email', 'birthday', 'gender', 'highestDegreeEarned', 'countryISO2', 'admissionsQuiz']


5. Contoh agregasi: menghitung pelamar per negara (hanya 3 hasil pertama)

In [17]:
# Aggregation: count applicants by country (first 3 results)
pipeline = [
    {"$group": {"_id": "$countryISO2", "count": {"$count": {}}}},
    {"$limit": 3}
]
sample_agg = list(demo_collection.aggregate(pipeline))
print("Sample aggregation result:")
for doc in sample_agg:
    print(doc)

Sample aggregation result:
{'_id': 'ZW', 'count': 94}
{'_id': 'MU', 'count': 3}
{'_id': 'TN', 'count': 16}


## Demo pandas dan Plotly

1. Menghitung usia pelamar dari kolom birthday sampai waktu sekarang ($$NOW)

In [18]:
# Aggregation: compute age in years for first 3 applicants
pipeline_age = [
    {
        "$project": {
            "years": {
                "$dateDiff": {
                    "startDate": "$birthday",
                    "endDate": "$$NOW",
                    "unit": "year",
                }
            }
        }
    },
    {"$limit": 3}
]
sample_ages = list(demo_collection.aggregate(pipeline_age))
print("Sample age computation:")
for doc in sample_ages:
    print(doc)

Sample age computation:
{'_id': ObjectId('6ae334b4c93055569870c3d2'), 'years': 28}
{'_id': ObjectId('6ae334b4c93055569870c3d3'), 'years': 27}
{'_id': ObjectId('6ae334b4c93055569870c3d4'), 'years': 32}


2. Mengubah hasil agregasi menjadi DataFrame

In [19]:
import pandas as pd

# Convert aggregation result to DataFrame
df_demo = pd.DataFrame(sample_ages)
print(df_demo)

                        _id  years
0  6ae334b4c93055569870c3d2     28
1  6ae334b4c93055569870c3d3     27
2  6ae334b4c93055569870c3d4     32


3. Contoh grafik batang horizontal sederhana (data buah: Apple, Banana, Cherry)

In [20]:
import plotly.express as px

# Toy bar chart
toy_data = pd.DataFrame({
    "fruit": ["Apple", "Banana", "Cherry"],
    "count": [10, 7, 3]
})
fig = px.bar(
    toy_data,
    x="count",
    y="fruit",
    orientation="h",
    title="Toy Bar Chart"
)
fig.update_layout(xaxis_title="Count", yaxis_title="Fruit")
fig.show()

## Persiapan

1. Perintah instalasi library (dikomentari)

In [21]:
# %pip install country_converter

2. Demo konversi kode ISO2 ke nama negara dan ISO3 (NG → Nigeria → NGA)

In [22]:
from country_converter import CountryConverter

cc = CountryConverter()

# Convert ISO2 codes to short names
print(cc.convert("NG", to="name_short"))  # Nigeria
print(cc.convert("IN", to="name_short"))  # India

# Convert ISO2 to ISO3 (needed for Plotly choropleth)
print(cc.convert("NG", to="ISO3"))  # NGA

Nigeria
India
NGA


3. Demo mencetak dictionary dengan rapi

In [23]:
from pprint import PrettyPrinter

pp = PrettyPrinter(indent=2)
pp.pprint({"name": "Alice", "scores": [90, 85, 92]})

{'name': 'Alice', 'scores': [90, 85, 92]}


4. Mengumpulkan semua import utama di satu tempat

In [24]:
from pprint import PrettyPrinter

import pandas as pd
import plotly.express as px
from country_converter import CountryConverter
from pymongo import MongoClient

5. Membuat objek pretty printer

In [25]:
pp = PrettyPrinter(indent = 2)
print("pp type:", type(pp))

pp type: <class 'pprint.PrettyPrinter'>


6. Membuat koneksi MongoDB untuk analisis sebenarnya

In [26]:
client = MongoClient(host=MONGODB_HOST, port=27017)
print("client type:", type(client))

client type: <class 'pymongo.synchronous.mongo_client.MongoClient'>


7. Menampilkan daftar database dengan rapi

In [27]:
# list available databases using pp
pp.pprint(client.list_database_names())

['admin', 'air-quality', 'config', 'local', 'wqu-abtest']


8. Mengakses koleksi ds-applicants sebagai ds_app

In [28]:
db = client["wqu-abtest"]
ds_app = db["ds-applicants"]
print("ds_app type:", type(ds_app))

ds_app type: <class 'pymongo.synchronous.collection.Collection'>


9. Validasi bahwa pp, client, dan ds_app bertipe benar

In [29]:
assert str(type(pp)) == "<class 'pprint.PrettyPrinter'>", (
    f"Expected pp to be a PrettyPrinter, got {type(pp)}"
)
assert "MongoClient" in str(type(client)), (
    f"Expected client to be a MongoClient, got {type(client)}"
)
assert ds_app.name == "ds-applicants", (
    f"Expected collection name 'ds-applicants', got '{ds_app.name}'"
)
print("All checks passed!")

All checks passed!


## Eksplorasi dataset

1. Menyimpan jumlah dokumen (5.025)

In [30]:
n_documents = ds_app.count_documents({})  # count all documents with an empty filter
print("Documents in 'ds-applicants':", n_documents)

Documents in 'ds-applicants': 5025


2. Menampilkan satu dokumen contoh secara rapi

In [31]:
result = ds_app.find_one({})  # retrieve a single document
pp.pprint(result)

{ '_id': ObjectId('6ae334b4c93055569870c3d2'),
  'admissionsQuiz': 'incomplete',
  'birthday': datetime.datetime(1998, 4, 29, 0, 0),
  'countryISO2': 'GB',
  'createdAt': datetime.datetime(2022, 5, 13, 15, 2, 44),
  'email': 'terry.hassler28@yahow.com',
  'firstName': 'Terry',
  'gender': 'male',
  'highestDegreeEarned': "Bachelor's degree",
  'lastName': 'Hassler'}


3. Validasi jumlah dokumen dan keberadaan field penting

In [32]:
assert isinstance(n_documents, int) and n_documents > 0, (
    f"Expected n_documents to be a positive integer, got {n_documents}"
)
assert isinstance(result, dict), (
    f"Expected result to be a dict, got {type(result)}"
)
expected_keys = {"countryISO2", "birthday", "highestDegreeEarned"}
actual_keys = set(result.keys())
assert expected_keys.issubset(actual_keys), (
    f"Expected document to contain keys {expected_keys}, "
    f"but got {actual_keys}"
)
print(f"Collection has {n_documents} documents. All checks passed!")

Collection has 5025 documents. All checks passed!


## Analisis negara pelamar

1. Menghitung jumlah pelamar per countryISO2

In [33]:
result = ds_app.aggregate(
    [{"$group": {"_id": "$countryISO2", "count": {"$count": {}}}}]
)
print("result type:", type(result))

result type: <class 'pymongo.synchronous.command_cursor.CommandCursor'>


2. Membuat DataFrame df_nationality (139 negara), diurutkan dari yang paling sedikit

In [34]:
df_nationality = (
    pd.DataFrame(result)
    .rename({"_id": "country_iso2"}, axis=1) # rename "_id" to "country_iso2"
    .sort_values("count")
)
print("df_nationality shape:", df_nationality.shape)
df_nationality.head()

df_nationality shape: (139, 2)


,country_iso2,count
0,CL,1
15,DJ,1
27,GY,1
21,GA,1
24,TM,1


3. Menambah kolom country_name (nama negara)

In [35]:
cc = CountryConverter()
df_nationality["country_name"] = cc.convert(
    df_nationality["country_iso2"], # pass the column with ISO2 codes
    to="name_short" # convert to short names
)
print("df_nationality shape:", df_nationality.shape)
df_nationality.head()

df_nationality shape: (139, 3)


,country_iso2,count,country_name
0,CL,1,Chile
15,DJ,1,Djibouti
27,GY,1,Guyana
21,GA,1,Gabon
24,TM,1,Turkmenistan


4. Grafik batang 10 negara dengan pelamar terbanyak (jumlah)

In [36]:
fig = px.bar(
    data_frame=df_nationality.tail(10), # last 10 rows
    x="count",
    y="country_name",
    orientation="h",
    title="DS Applicants by Country",
)
fig.update_layout(
    xaxis_title="Frequency [count]",
    yaxis_title="Country"
)
fig.show()

5. Menambah kolom count_pct (proporsi)

In [37]:
# compute proportion: count divided by total
df_nationality["count_pct"] = (
    df_nationality["count"] / df_nationality["count"].sum()
)
print("df_nationality shape:", df_nationality.shape)
df_nationality.head()

df_nationality shape: (139, 4)


,country_iso2,count,country_name,count_pct
0,CL,1,Chile,0.000199
15,DJ,1,Djibouti,0.000199
27,GY,1,Guyana,0.000199
21,GA,1,Gabon,0.000199
24,TM,1,Turkmenistan,0.000199


6. Grafik batang 10 negara teratas dalam bentuk persentase

In [38]:
fig = px.bar(
    data_frame=df_nationality.tail(10),
    x="count_pct", # use the percentage column
    y="country_name",
    orientation="h",
    title="DS Applicants by Country",
)
fig.update_layout(
    xaxis_title="Frequency [%]",
    yaxis_title="Country"
)
fig.show()

7. Menambah kolom country_iso3 untuk peta

In [39]:
df_nationality["country_iso3"] = cc.convert(
    df_nationality["country_iso2"],  # pass the column with ISO2 codes
    to="ISO3"  # convert to three-letter ISO codes
)
print("df_nationality shape:", df_nationality.shape)
df_nationality.head()

df_nationality shape: (139, 5)


,country_iso2,count,country_name,count_pct,country_iso3
0,CL,1,Chile,0.000199,CHL
15,DJ,1,Djibouti,0.000199,DJI
27,GY,1,Guyana,0.000199,GUY
21,GA,1,Gabon,0.000199,GAB
24,TM,1,Turkmenistan,0.000199,TKM


8. Peta dunia (choropleth) sebaran pelamar per negara, skema warna oranye

In [40]:
fig = px.choropleth(
    data_frame=df_nationality,
    locations="country_iso3", # column with ISO3 codes
    color="count",
    projection="natural earth",
    color_continuous_scale=px.colors.sequential.Oranges, # check colors from px.colors.sequential
    title="DS Applicants: Nationalities",
)
fig.show()

9. Validasi kolom lengkap dan total proporsi ≈ 1

In [41]:
assert "country_iso2" in df_nationality.columns, (
    "df_nationality is missing the 'country_iso2' column"
)
assert "count" in df_nationality.columns, (
    "df_nationality is missing the 'count' column"
)
assert "country_name" in df_nationality.columns, (
    "df_nationality is missing the 'country_name' column"
)
assert "count_pct" in df_nationality.columns, (
    "df_nationality is missing the 'count_pct' column"
)
assert "country_iso3" in df_nationality.columns, (
    "df_nationality is missing the 'country_iso3' column"
)
pct_sum = df_nationality["count_pct"].sum()
assert abs(pct_sum - 1.0) < 0.01, (
    f"Expected count_pct to sum to ~1.0, got {pct_sum:.4f}"
)
print(
    f"df_nationality has {df_nationality.shape[0]} countries "
    f"and {df_nationality.shape[1]} columns. All checks passed!"
)

df_nationality has 139 countries and 5 columns. All checks passed!


## Analisis usia pelamar

1. Menghitung usia seluruh pelamar

In [42]:
result = ds_app.aggregate(
    [
        {
            "$project": {
                "years": {
                    "$dateDiff": {
                        "startDate": "$birthday",
                        "endDate": "$$NOW",
                        "unit": "year",
                    }
                }
            }
        }
    ]
)
print("result type:", type(result))

result type: <class 'pymongo.synchronous.command_cursor.CommandCursor'>


2. Mengambil kolom usia sebagai pd.Series (5.025 nilai)

In [43]:
# convert result to DataFrame and extract the "years" column
ages = pd.DataFrame(result)["years"]
print("ages type:", type(ages))
print("ages shape:", ages.shape)
ages.head()

ages type: <class 'pandas.Series'>
ages shape: (5025,)


0    28
1    27
2    32
3    42
4    36
Name: years, dtype: int64

3. Histogram distribusi usia pelamar

In [44]:
fig = px.histogram(
    x=ages,
    nbins=20,
    title="Distribution of DS Applicant Ages",
)
fig.update_layout(
    xaxis_title="Age",
    yaxis_title="Frequency [count]"
)
fig.show()

4. Validasi usia wajar (rentang 22-76 tahun)

In [45]:
assert isinstance(ages, pd.Series), (
    f"Expected ages to be a pd.Series, got {type(ages)}"
)
assert len(ages) > 0, "ages Series is empty"
assert ages.min() >= 0, (
    f"Expected all ages >= 0, but min age is {ages.min()}"
)
assert ages.max() < 150, (
    f"Expected reasonable max age, but got {ages.max()}"
)
print(
    f"ages has {len(ages)} values, "
    f"range [{ages.min()}, {ages.max()}]. All checks passed!"
)

ages has 5025 values, range [22, 76]. All checks passed!


## Analisis tingkat pendidikan

1. Menghitung pelamar per jenjang pendidikan

In [46]:
result = ds_app.aggregate(
    [{"$group": {"_id": "$highestDegreeEarned", "count": {"$count": {}}}}]
)
print("result type:", type(result))

result type: <class 'pymongo.synchronous.command_cursor.CommandCursor'>


2. Mengubah hasil menjadi Series education

In [47]:
education = (
    pd.DataFrame(result)
    .rename({"_id": "highest_degree_earned"}, axis=1)
    .set_index("highest_degree_earned")
    .squeeze()
)
print("education type:", type(education))
print("education shape:", education.shape)
education

education type: <class 'pandas.Series'>
education shape: (5,)


highest_degree_earned
Some College (1-3 years)         612
Master's degree                  862
Bachelor's degree               2643
High School or Baccalaureate     832
Doctorate (e.g. PhD)              76
Name: count, dtype: int64

3. Mengurutkan pendidikan dari terendah (SMA) ke tertinggi (Doktor)

In [ ]:
def ed_sort(counts):
    """Sort array `counts` from lowest to highest degree."""
    degrees = [
        "High School or Baccalaureate",
        "Some College (1-3 years)",
        "Bachelor's degree",
        "Master's degree",
        "Doctorate (e.g. PhD)"
    ]
    # create a dict mapping degree name -> position
    mapping = {deg: i for i, deg in enumerate(degrees)}
    # build a list of positions for each entry in counts
    sort_order = [mapping[deg] for deg in counts]
    return sort_order
# INSERT
# INSERT
education.sort_index(key=ed_sort, inplace=True)
education

highest_degree_earned
High School or Baccalaureate     832
Some College (1-3 years)         612
Bachelor's degree               2643
Master's degree                  862
Doctorate (e.g. PhD)              76
Name: count, dtype: int64

4. Grafik batang horizontal tingkat pendidikan

In [49]:
fig = px.bar(
    x=education,
    y=education.index,
    orientation="h",
    title="DS Applicant Education Levels",
)
fig.update_layout(
    xaxis_title="Frequency [count]",
    yaxis_title="Highest Degree Earned"
)
fig.show()

5. Validasi 5 jenjang pendidikan dan urutannya benar

In [50]:
assert isinstance(education, pd.Series), (
    f"Expected education to be a pd.Series, got {type(education)}"
)
assert len(education) == 5, (
    f"Expected 5 education levels, got {len(education)}"
)
expected_order = [
    "High School or Baccalaureate",
    "Some College (1-3 years)",
    "Bachelor's degree",
    "Master's degree",
    "Doctorate (e.g. PhD)",
]
actual_order = list(education.index)
assert actual_order == expected_order, (
    f"Expected education index order:\n{expected_order}\n"
    f"Got:\n{actual_order}"
)
assert education.sum() > 0, "education counts should be positive"
print(
    f"education has {len(education)} levels, "
    f"total count: {education.sum()}. All checks passed!"
)

education has 5 levels, total count: 5025. All checks passed!


Kesimpulan Singkat

- Proyek ini adalah analisis eksplorasi data (EDA) terhadap 5.025 pelamar program Data Science yang tersimpan di MongoDB. 
- Alurnya: data diambil dengan aggregation pipeline MongoDB, diolah dengan pandas, lalu divisualisasikan dengan Plotly. 
- Temuan utamanya:
    - Asal negara: 
        - Pelamar berasal dari 139 negara. 
        - Nigeria menempati posisi teratas, disusul Pakistan, India, Kenya, dan Bangladesh. - Pelamar terkonsentrasi di negara-negara berkembang, terutama Afrika dan Asia Selatan.
    - Usia: 
        - Rentang usia 22-76 tahun, dengan sebagian besar pelamar berada di usia sekitar 20-an akhir hingga 30-an awal.
    - Pendidikan: 
        - Mayoritas pelamar bergelar Sarjana (Bachelor’s) (2.643 orang, sekitar 53%), diikuti Master (862), SMA/Baccalaureate (832), Some College (612), dan Doktor (76).

- Secara keseluruhan, program ini menarik peminat dari kalangan lulusan sarjana usia muda dari berbagai negara berkembang. 
- Notebook ini juga menerapkan praktik yang baik, seperti validasi dengan assert di tiap tahap, dan menjadi dasar yang kuat untuk analisis lanjutan, misalnya A/B test (sesuai nama database wqu-abtest) atau hubungan antara karakteristik pelamar dengan penyelesaian admissions quiz.